# 06_state_extraction_and_plots
Run NLP vs LLM state extraction, evaluate, and plot comparisons.

In [1]:
# If needed:
# !pip install -U pandas numpy scikit-learn matplotlib jsonschema openai
print('Notebook ready.')

Notebook ready.


In [2]:
from pathlib import Path
import json

data_dir = Path('../data/synthetic_data')
sample = sorted(data_dir.glob('transcript_*.json'))[0]
with open(sample, 'r', encoding='utf-8') as f:
    tr = json.load(f)
print('Sample intent:', tr['gt_primary_intent'])
print('Scenario:', tr['gt_scenario_family'])
print('Tool failure:', tr['gt_tool_failure'])
print('Turn count:', tr['gt_turn_count'])

Sample intent: CARD_REPLACEMENT
Scenario: clarify_then_resolve
Tool failure: False
Turn count: 7


In [3]:
# Run NLP state extraction (offline)
!python ../src/state_extraction_pipeline.py --input-dir ../data/synthetic_data --output-dir ../outputs/state_nlp --provider nlp --limit 300

# Evaluate NLP extraction
!python ../src/eval_state_extraction.py --pred-dir ../outputs/state_nlp --output-dir ../results/state_nlp

Wrote 300 state predictions to ..\outputs\state_nlp
{
  "n_samples": 300,
  "provider": "nlp",
  "model": "heuristic-nlp",
  "primary_intent": {
    "accuracy": 0.38666666666666666,
    "macro_precision": 0.3879121730784135,
    "macro_recall": 0.26812578252751107,
    "macro_f1": 0.27831941995139503
  },
  "secondary_intents": {
    "precision": 0.18475073313782991,
    "recall": 0.38414634146341464,
    "f1": 0.2495049504950495,
    "exact_match": 0.26
  },
  "multi_intent_accuracy": 0.72,
  "tool_failure_accuracy": 1.0,
  "ambiguity": {
    "mae": 0.19311333333333333,
    "rmse": 0.2184024572511338,
    "pearson": 0.5223865839917812
  },
  "sentiment_overall": {
    "mae": 0.0,
    "rmse": 0.0,
    "pearson": 1.0
  },
  "turn_count_mae": 0.0,
  "failure_count_mae": 0.0,
  "schema_valid_rate": 1.0,
  "average_jaccard_secondary": 0.3444444444444445
}
Saved summary + rows to C:\Mtech project\mtech-voicebot-context-project\results\state_nlp


In [1]:
# Optional: run LLM extraction if you have API access
!python ../src/state_extraction_pipeline.py --input-dir ../data/synthetic_data --output-dir ../outputs/state_ollama --provider ollama --model qwen2.5:7b-instruct --limit 300
!python ../src/eval_state_extraction.py --pred-dir ../outputs/state_ollama --output-dir ../results/state_ollama
# print('LLM mode is optional; NLP mode is already runnable offline.')

Traceback (most recent call last):
  File "C:\Mtech project\mtech-voicebot-context-project\src\state_extraction_pipeline.py", line 565, in <module>
    main()
  File "C:\Mtech project\mtech-voicebot-context-project\src\state_extraction_pipeline.py", line 561, in main
    run_batch(args.input_dir, args.output_dir, args.provider, args.model, args.limit)
  File "C:\Mtech project\mtech-voicebot-context-project\src\state_extraction_pipeline.py", line 515, in run_batch
    llm_pred = extract_one(tr, provider=provider, model=model)
  File "C:\Mtech project\mtech-voicebot-context-project\src\state_extraction_pipeline.py", line 438, in extract_one
    "user": LLM_USER_PROMPT_TEMPLATE.format(
KeyError: '\n  "primary_intent"'
Traceback (most recent call last):
  File "C:\Mtech project\mtech-voicebot-context-project\src\eval_state_extraction.py", line 176, in <module>
    main()
  File "C:\Mtech project\mtech-voicebot-context-project\src\eval_state_extraction.py", line 163, in main
    summary, df

In [3]:
# Plot comparison for one or more methods
# Example:
!python ../src/plot_state_comparison.py \
  --summaries ../results/state_nlp/state_eval_summary.json ../results/state_ollama/state_eval_summary.json \
  --rows ../results/state_nlp/state_eval_rows.csv ../results/state_ollama/state_eval_rows.csv \
  --labels NLP LLM \
  --output-dir ../results/state_plots
# print('Use plot_state_comparison.py after you have one or more evaluation summaries.')

Saved plots and tables to C:\Mtech project\mtech-voicebot-context-project\results\state_plots
